 rolling averages

 QN: real-time scenario where you have a dataset of stock price data, and you want to calculate the rolling average stock price for each stock within a specific time window. 

 A rolling average, also known as a moving average, is a statistical calculation used in time series analysis to smooth out fluctuations in data over a specified period. It is a commonly used technique in various fields, including finance, economics, signal processing, and data analysis. The rolling average provides a more stable and less noisy representation of data, making it easier to identify trends and patterns.

Here's how a rolling average is calculated:

1. Select a Window Size: You choose a fixed window size (e.g., 3 days, 7 days, 30 days) that defines the number of data points to include in each average calculation.

2. Slide the Window: Starting from the first data point, you slide the window over the time series data one step at a time. At each step, you include the data points within the window and calculate their average.

3. Smoothed Data: The result of this calculation is a new time series, often referred to as the rolling average or moving average. Each point in the rolling average represents the average value of the data within the window at that point in time.

The rolling average is useful for various purposes:

1. Noise Reduction: It helps reduce random fluctuations or noise in time series data, making underlying trends more apparent.

2. Trend Identification: By smoothing the data, rolling averages make it easier to identify long-term trends, upward or downward movements, or cyclic patterns.

3. Seasonal Adjustment: In economics and finance, rolling averages are often used to seasonally adjust data by removing the seasonal components.

4. Forecasting: Rolling averages can be used as a basis for forecasting future values in time series analysis.

Example:
Let's say you have daily stock price data for a company, and you calculate a 7-day rolling average. The rolling average for a particular day would be the average of the stock prices for that day and the previous six days. This smooths out daily fluctuations, making it easier to see the overall trend in the stock's price.

In PySpark, as shown in a previous response, you can use window functions to calculate rolling averages within a specified window size for time-based data analysis.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import col, avg

# Create a Spark session
spark = SparkSession.builder.appName("RealTimeStockAnalysis").getOrCreate()

# Sample stock price dataset
data = [
    ("AAPL", "2023-08-26 09:00:00", 150.0),
    ("AAPL", "2023-08-26 09:30:00", 152.0),
    ("AAPL", "2023-08-26 10:00:00", 153.5),
    ("AAPL", "2023-08-26 10:30:00", 155.0),
    ("AAPL", "2023-08-26 11:00:00", 154.5),
    ("AAPL", "2023-08-26 11:30:00", 156.0),
    ("AAPL", "2023-08-26 12:00:00", 157.0),
    ("AAPL", "2023-08-26 12:30:00", 158.0),
    ("AAPL", "2023-08-26 13:00:00", 158.5),
    ("AAPL", "2023-08-26 13:30:00", 160.0),
    ("AAPL", "2023-08-26 14:00:00", 159.0),
]

# Create a DataFrame from the data
schema = ["symbol", "timestamp", "price"]
df = spark.createDataFrame(data, schema=schema)

# Convert the timestamp column to a timestamp type
df = df.withColumn("timestamp", col("timestamp").cast("timestamp"))

# Define a window specification for a rolling 3-hour window partitioned by symbol and ordered by timestamp
window_spec = Window.partitionBy("symbol").orderBy("timestamp").rowsBetween(-3, 0)

# Calculate the rolling average stock price within the 3-hour window
df = df.withColumn("rolling_avg_price", avg("price").over(window_spec))

# Show the result
df.show()

+------+-------------------+-----+------------------+
|symbol|          timestamp|price| rolling_avg_price|
+------+-------------------+-----+------------------+
|  AAPL|2023-08-26 09:00:00|150.0|             150.0|
|  AAPL|2023-08-26 09:30:00|152.0|             151.0|
|  AAPL|2023-08-26 10:00:00|153.5|151.83333333333334|
|  AAPL|2023-08-26 10:30:00|155.0|           152.625|
|  AAPL|2023-08-26 11:00:00|154.5|            153.75|
|  AAPL|2023-08-26 11:30:00|156.0|            154.75|
|  AAPL|2023-08-26 12:00:00|157.0|           155.625|
|  AAPL|2023-08-26 12:30:00|158.0|           156.375|
|  AAPL|2023-08-26 13:00:00|158.5|           157.375|
|  AAPL|2023-08-26 13:30:00|160.0|           158.375|
|  AAPL|2023-08-26 14:00:00|159.0|           158.875|
+------+-------------------+-----+------------------+

